In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Data augmentation
transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),      # Randomly crops the image (with padding) -> improves robustness
    transforms.RandomHorizontalFlip(),         # Randomly flips images horizontally -> prevents overfitting
    transforms.ToTensor(),                     # Converts images from PIL to PyTorch tensors
    transforms.Normalize((0.5,), (0.5,))       # Normalizes pixels to mean=0.5, std=0.5 -> scale to [-1,1]
])


train_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=False, transform=transform
)   # Loads CIFAR-10 training data with augmentation

test_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=False, transform=transform
)   # Loads CIFAR-10 test data (same transform, no augmentation needed)

train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=128, shuffle=True
)   # Creates mini-batches (128 samples), shuffling for training

test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=128, shuffle=False
)   # Creates mini-batches for testing (no shuffle)


# Involution 2D Layer
class Involution2D(nn.Module):
    def __init__(self, channels, kernel_size=3, stride=1, reduction_ratio=4): #channels = input feature channels
        # kernel_size = size of local patch, stride = downsampling step, reduction_ratio = for channel reduction
        super(Involution2D, self).__init__()
        self.channels = channels
        self.kernel_size = kernel_size
        self.stride = stride

        self.reduce_channels = channels // reduction_ratio   #Reduces the number of channels (to save computation when generating kernels).
        self.conv1 = nn.Conv2d(channels, self.reduce_channels, 1)  #First 1x1 convolution: compress channels.
        self.bn = nn.BatchNorm2d(self.reduce_channels)   #Batch Normalization: stabilizes training.
        self.conv2 = nn.Conv2d(self.reduce_channels, kernel_size * kernel_size, 1)  #Second 1x1 convolution: outputs a kernel of size (k*k) per pixel location.
        self.unfold = nn.Unfold(kernel_size=kernel_size, padding=kernel_size // 2, stride=stride)  #nn.Unfold: extracts local sliding patches from input (like im2col).

    def forward(self, x):
        batch_size, channels, height, width = x.shape  #Get input tensor dimensions.

        weight = self.conv1(x)
        weight = self.bn(weight)
        weight = self.conv2(weight)   #Generate adaptive weights (kernels) from the input.
        weight = weight.view(batch_size, 1, self.kernel_size * self.kernel_size, height, width)   #Reshape so each pixel has a (k*k) kernel.

        x_unfold = self.unfold(x)
        x_unfold = x_unfold.view(batch_size, channels, self.kernel_size * self.kernel_size, height, width)  #Extract local patches around each pixel.

        out = (weight * x_unfold).sum(dim=2)   #Apply generated weights (kernels) to local patches -> like convolution but adaptive.
        if self.stride > 1:    #Downsampling if stride > 1.
            out = out[:, :, ::self.stride, ::self.stride]
        return out   #Output processed feature map.

# Involutional Neural Network
class InvolutionNet(nn.Module):
    def __init__(self):
        super(InvolutionNet, self).__init__()
        self.block1 = nn.Sequential(    #Block 1: input image (3 channels) -> project to 64 -> apply Involution -> normalize -> downsample.
            nn.Conv2d(3, 64, 1), nn.BatchNorm2d(64), nn.ReLU(),
            Involution2D(64),
            nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.block2 = nn.Sequential(    #Block 2: 64 -> 128 channels with Involution.
            nn.Conv2d(64, 128, 1), nn.BatchNorm2d(128), nn.ReLU(),
            Involution2D(128),
            nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.block3 = nn.Sequential(   #Block 3: 128 -> 256 channels, downsample further.
            nn.Conv2d(128, 256, 1), nn.BatchNorm2d(256), nn.ReLU(),
            Involution2D(256),
            nn.BatchNorm2d(256), nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.fc = nn.Sequential(    #Fully Connected classifier: flatten -> dense layer -> dropout -> final 10-class output.
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.fc(x)
        return x

# Initialize model
model = InvolutionNet().to(device)
print(model)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()   # Loss function for classification
optimizer = optim.Adam(model.parameters(), lr=0.001)   # Adam optimizer
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50) 
# Scheduler -> smoothly decreases learning rate using cosine curve

# Training
num_epochs = 50
for epoch in range(num_epochs):
    model.train()          # Enable training mode (dropout, batchnorm)
    running_loss = 0
    for images, labels in train_loader:   # Loop over mini-batches
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)           # Forward pass
        loss = criterion(outputs, labels) # Compute loss

        optimizer.zero_grad()  # Reset gradients
        loss.backward()        # Backpropagation
        optimizer.step()       # Update parameters
        running_loss += loss.item()   # Track training loss

    scheduler.step()   # Update learning rate
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss / len(train_loader):.4f}")

# Evaluation
model.eval()   # Switch to evaluation mode (disables dropout, BN in inference mode)
correct = 0
total = 0
with torch.no_grad():   # Disable gradient calculation (faster inference)
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)           # Forward pass
        _, predicted = torch.max(outputs, 1) # Get class with highest probability
        total += labels.size(0)           # Count samples
        correct += (predicted == labels).sum().item()  # Count correct predictions


acc = 100 * correct / total
print(f"InvolutionNet Accuracy on CIFAR-10: {acc:.2f}%")


Device: cuda
InvolutionNet(
  (block1): Sequential(
    (0): Conv2d(3, 64, kernel_size=(1, 1), stride=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Involution2D(
      (conv1): Conv2d(64, 16, kernel_size=(1, 1), stride=(1, 1))
      (bn): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(16, 9, kernel_size=(1, 1), stride=(1, 1))
      (unfold): Unfold(kernel_size=3, dilation=1, padding=1, stride=1)
    )
    (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block2): Sequential(
    (0): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1))
    (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Involution2D(
      (conv1): Conv2d(128, 32, kernel_size=(1, 1), stride=

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Data augmentation and normalization for CIFAR-10
transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.4914, 0.4822, 0.4465], 
        std=[0.2023, 0.1994, 0.2010]
    )
])

train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=False, transform=transform)
test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=False, transform=transform)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=4, pin_memory=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=4, pin_memory=True)

# Involution 2D Layer
class Involution2D(nn.Module):
    def __init__(self, channels, kernel_size=3, stride=1, reduction_ratio=4):
        super(Involution2D, self).__init__()
        self.channels = channels
        self.kernel_size = kernel_size
        self.stride = stride

        self.reduce_channels = channels // reduction_ratio
        self.conv1 = nn.Conv2d(channels, self.reduce_channels, 1)
        self.bn = nn.BatchNorm2d(self.reduce_channels)
        self.conv2 = nn.Conv2d(self.reduce_channels, kernel_size * kernel_size, 1)
        self.unfold = nn.Unfold(kernel_size=kernel_size, padding=kernel_size // 2, stride=stride)

    def forward(self, x):
        batch_size, channels, height, width = x.shape

        weight = self.conv1(x)
        weight = self.bn(weight)
        weight = self.conv2(weight)
        weight = weight.view(batch_size, 1, self.kernel_size * self.kernel_size, height, width)

        x_unfold = self.unfold(x)
        x_unfold = x_unfold.view(batch_size, channels, self.kernel_size * self.kernel_size, height, width)

        out = (weight * x_unfold).sum(dim=2)
        if self.stride > 1:
            out = out[:, :, ::self.stride, ::self.stride]
        return out

# Involutional Neural Network with 3x3 convs
class InvolutionNet(nn.Module):
    def __init__(self):
        super(InvolutionNet, self).__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            Involution2D(64),
            nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            Involution2D(128),
            nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.block3 = nn.Sequential(
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            Involution2D(256),
            nn.BatchNorm2d(256), nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 512), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.fc(x)
        return x

# Initialize model
model = InvolutionNet().to(device)
print(model)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)

# Training loop
num_epochs = 100
for epoch in range(num_epochs):
    model.train()
    running_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    scheduler.step()
    avg_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}")

    # Optional: Evaluate every 10 epochs
    if (epoch + 1) % 10 == 0:
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        acc = 100 * correct / total
        print(f"Test Accuracy after epoch {epoch+1}: {acc:.2f}%")

# Final evaluation
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

acc = 100 * correct / total
print(f"Final InvolutionNet Accuracy on CIFAR-10: {acc:.2f}%")


Device: cuda
InvolutionNet(
  (block1): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Involution2D(
      (conv1): Conv2d(64, 16, kernel_size=(1, 1), stride=(1, 1))
      (bn): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(16, 9, kernel_size=(1, 1), stride=(1, 1))
      (unfold): Unfold(kernel_size=3, dilation=1, padding=1, stride=1)
    )
    (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block2): Sequential(
    (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Involution2D(
      (conv1): Conv2d(128,